# 08 · Bronze · Site (canal e-commerce)
Ingere os 4 arquivos do site: `pedidos`, `pedido_itens` e `eventos`, definidos no contrato com a API (`SCHEMA_PIPELINE_v1`), e `avaliacoes`, no formato proposto para a v1.1 (D25).

Duas origens, **com as mesmas colunas**:
- `landing/site/historico/`: histórico sintético até o corte de 2026-11-18T00:00:00Z (`gera_site.py`, D23; `gera_avaliacoes.py`, D25);
- `landing/site/api/`: exportação incremental da API do Caio (`npm run export:novos`), um arquivo por execução, com data e hora no nome.

**Tabelas geradas:** `site_pedidos`, `site_pedido_itens`, `site_eventos`, `site_avaliacoes`

### Ingestão
Mesmo padrão das outras bronzes (D05 e D06): tudo como texto, `_arquivo_origem` e `_ingerido_em`, carga completa.
- **O caminho usa `*` no lugar da origem** (`site/*/pedidos/`): a mesma consulta lê o histórico e todos os arquivos que a API mandar depois.
- `_arquivo_origem` guarda o **caminho completo**, e não só o nome do arquivo: é por ele que se sabe se a linha veio do histórico ou da API.
- **Nada é deduplicado aqui:** um pedido cancelado chega de novo, como nova versão. Quem escolhe a versão mais recente é a silver.

**Equivalente no BigQuery:** tabela externa com `format = 'CSV'`, `skip_leading_rows = 1` e uma lista de URIs, uma por origem (`gs://<bucket>/site/historico/pedidos/*` e `gs://<bucket>/site/api/pedidos/*`); `_FILE_NAME` traz o caminho do arquivo.

In [0]:
CREATE OR REPLACE TABLE horizonte.bronze.site_pedidos AS
SELECT * EXCEPT (_rescued_data),
       _metadata.file_path AS _arquivo_origem,
       current_timestamp() AS _ingerido_em
FROM read_files('/Volumes/horizonte/bronze/landing/site/*/pedidos/',
                format => 'csv', header => true, inferSchema => false);

In [0]:
CREATE OR REPLACE TABLE horizonte.bronze.site_pedido_itens AS
SELECT * EXCEPT (_rescued_data),
       _metadata.file_path AS _arquivo_origem,
       current_timestamp() AS _ingerido_em
FROM read_files('/Volumes/horizonte/bronze/landing/site/*/pedido_itens/',
                format => 'csv', header => true, inferSchema => false);

In [0]:
CREATE OR REPLACE TABLE horizonte.bronze.site_eventos AS
SELECT * EXCEPT (_rescued_data),
       _metadata.file_path AS _arquivo_origem,
       current_timestamp() AS _ingerido_em
FROM read_files('/Volumes/horizonte/bronze/landing/site/*/eventos/',
                format => 'csv', header => true, inferSchema => false);

### Avaliações
Mesmo padrão, com duas opções a mais: título e comentário são digitados pelo cliente e podem ter quebras de linha e aspas. Por isso `multiLine => true` e `escape => '"'`, como nas avaliações do Olist (notebook 01). **BigQuery:** `allow_quoted_newlines = true` na tabela externa.

In [0]:
CREATE OR REPLACE TABLE horizonte.bronze.site_avaliacoes AS
SELECT * EXCEPT (_rescued_data),
       _metadata.file_path AS _arquivo_origem,
       current_timestamp() AS _ingerido_em
FROM read_files('/Volumes/horizonte/bronze/landing/site/*/avaliacoes/',
                format => 'csv', header => true, inferSchema => false,
                multiLine => true, escape => '"');

In [0]:
SELECT 'site_pedidos' AS tabela, COUNT(*) AS linhas, COUNT(DISTINCT _arquivo_origem) AS arquivos
FROM horizonte.bronze.site_pedidos                                                    -- 20.180 (só histórico)
UNION ALL SELECT 'site_pedido_itens', COUNT(*), COUNT(DISTINCT _arquivo_origem)
FROM horizonte.bronze.site_pedido_itens                                               -- 28.407
UNION ALL SELECT 'site_eventos', COUNT(*), COUNT(DISTINCT _arquivo_origem)
FROM horizonte.bronze.site_eventos                                                    -- 1.816.833
UNION ALL SELECT 'site_avaliacoes', COUNT(*), COUNT(DISTINCT _arquivo_origem)
FROM horizonte.bronze.site_avaliacoes;                                                -- 2.584